# LOGAN: what the structural features add beyond ORF length

**Draft caption.** Average precision (a–d) and ROC AUC (e–h) of gradient-boosted classifiers separating complete Prodigal+ ORFs from complete Prodigal− ORFs, for four negative sets: all Prodigal−, antisense shadows, frameshift shadows and intergenic ORFs. Feature sets are nested on ORF length (dashed line: length alone). Five-fold cross-validation grouped by accession; metrics are population-weighted (sample weight = 1/sampling rate) over 9.5 million sampled complete ORFs from all 90 shards; error bars, ± SD across folds.

**Outputs:** `../images/figL02_models_beyond_length.pdf` (vector) and `../images/figL02_models_beyond_length.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL02_models_overall.tsv.gz` (28 rows; sha256 `4350f8ec90fd…`): AUC / AP ± fold SD, nested feature sets, 4 contrasts. Source on Setonix (`$LOGAN_ROOT`): `qc/analysis_models_final/models_overall.tsv`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL02_models_beyond_length.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
r = pd.read_csv(DATA / "figL02_models_overall.tsv.gz", sep="\t")
SET_LABEL = {"length": "length", "length+aa": "+ AA entropy", "length+3di": "+ 3Di entropy",
             "length+12st": "+ 12-state entropy", "length+mi": "+ 3Di–12st MI",
             "length+3di+12st+mi": "+ 3Di, 12st, MI", "length+aa+3di+12st+mi": "+ all four"}
ORDER = list(SET_LABEL)
CONTRASTS = [("vs all Prodigal-", "all Prodigal−"), ("vs antisense shadows", "antisense shadows"),
             ("vs frameshift shadows", "frameshift shadows"), ("vs intergenic", "intergenic")]
fig, axes = plt.subplots(2, 4, figsize=(19, 8.5), sharey=True)
letters = iter("abcdefgh")
for row, (metric, sd, lab) in enumerate((("avg_precision", "ap_fold_sd", "average precision"),
                                         ("roc_auc", "auc_fold_sd", "ROC AUC"))):
    for c, (contrast, ctxt) in enumerate(CONTRASTS):
        ax = axes[row, c]
        d = r[r.contrast == contrast].set_index("features").loc[ORDER]
        y = np.arange(len(ORDER))
        ax.barh(y, d[metric], xerr=d[sd], color=[SLOTS[0]] + [SLOTS[1]] * 6, height=0.65,
                error_kw={"ecolor": INK2, "elinewidth": 1})
        ax.axvline(d.loc["length", metric], color=INK2, lw=1, ls="--")
        lo = max(0.0, d[metric].min() - 0.08) if metric == "avg_precision" else max(0.9, d[metric].min() - 0.01)
        ax.set_xlim(lo, 1.0)
        for yi, v in zip(y, d[metric]):
            ax.text(v + 0.002, yi, f"{v:.3f}", va="center", fontsize=8.5, color=INK2)
        ax.set_xlabel(lab)
        if row == 0:
            ax.set_title(f"Prodigal+ vs {ctxt}", loc="left", fontsize=11)
        ax.set_yticks(y, [SET_LABEL[s] for s in ORDER])
        ax.invert_yaxis()
        panel(ax, next(letters), x=-0.02 if c else -0.32)
fig.tight_layout()
save(fig, "figL02_models_beyond_length")